In [16]:
import numpy as np

import usf

import os
usf_folders = [#'/home/matt/geophysics_data/TEM/sTEM_data/2025_0227/USF_Files_dawa river/',
               #'/home/matt/geophysics_data/TEM/sTEM_data/2025_0226/USF_Files_Dawa River Channel/',
            #    'E:/sTEM_data/2025_0302/USF_Files_dawa river 3/',
                # 'C:/Users/au646858/Downloads/2025_0622_profiler_sTEM53/USF_Files_profile_day_4_DD_Boren_Aliy_river/',
                'C:/Users/au646858/Downloads/USF_Files_dawa river/',

            #    '/home/matt/geophysics_data/TEM/sTEM_data/2025_0302/USF_Files_dawa river 3/'
               ]

sweeps = []
for usf_folder in usf_folders:
    usf_files = [usf_folder+f for f in os.listdir(usf_folder) if f.endswith('.usf')]
    for usf_file in usf_files:        
        sweeps += usf.Sweep.from_usf(usf_file, combine=False)
points = set([x.points for x in sweeps])

mega_sweeps = []
for pnts in points:
    mega_sweeps.append(usf.Sweep.combine_sweeps([swp for swp in sweeps if swp.points==pnts]))
lm,hm = mega_sweeps
import pandas as pd


site_id = list(range(len(lm.location)))
site_id = site_id+site_id

# Create a DataFrame with the specified columns
data = {
    'date'     : [int(x) for x in lm.date] + [int(x) for x in hm.date],
    'time'     : [float(x) for x in lm.daytime]+[float(x) for x in hm.daytime],
    'latitude' : [float(x[1]) for x in lm.location]+[float(x[1]) for x in hm.location],
    'longitude': [float(x[0]) for x in lm.location]+[float(x[0]) for x in hm.location],
    'elevation': [float(x[2]) for x in lm.location]+[float(x[2]) for x in hm.location],
    'system_id': [0 for _ in lm.date]+[1 for _ in hm.date],
    'site_id'  : site_id,
}

df = pd.DataFrame(data)
df = df.sort_values(by=['site_id','time'])

In [22]:
import pandas as pd
import numpy as np

def create_measurement_table(sweeps):
    """Create a table with measurement system parameters"""
    
    table_data = []
    
    for sweep in sweeps:
        # Extract transmitter and receiver coordinates
        # Assuming location contains [lon, lat, elevation] and tx/rx are at same location
        tx_x = float(sweep.location[0]) if hasattr(sweep, 'location') else np.nan
        tx_y = float(sweep.location[1]) if hasattr(sweep, 'location') else np.nan 
        tx_z = float(sweep.location[2]) if hasattr(sweep, 'location') else np.nan
        
        # For now, assuming tx and rx are co-located (common in TEM)
        rx_x = tx_x
        rx_y = tx_y  
        rx_z = tx_z
        
        # Number of gates
        n_gates = len(sweep.time) if hasattr(sweep, 'time') else np.nan
        
        # Gate timing
        t_open = float(sweep.time[0]) if hasattr(sweep, 'time') and len(sweep.time) > 0 else np.nan
        t_close = float(sweep.time[-1]) if hasattr(sweep, 'time') and len(sweep.time) > 0 else np.nan
        
        # Current
        current = float(sweep.current) if hasattr(sweep, 'current') else np.nan
        
        # Waveform parameters
        waveform_time = sweep.tx_ramp_time.tolist() if hasattr(sweep, 'tx_ramp_time') else []
        waveform_amplitude = sweep.tx_ramp.tolist() if hasattr(sweep, 'tx_ramp') else []
        
        # Filter parameters
        n_filters = len(sweep.low_pass) if hasattr(sweep, 'low_pass') else np.nan
        f_cut = float(sweep.low_pass[0]) if hasattr(sweep, 'low_pass') and len(sweep.low_pass) > 0 else np.nan
        
        table_data.append({
            'tx_x': tx_x,
            'tx_y': tx_y, 
            'tx_z': tx_z,
            'rx_x': rx_x,
            'rx_y': rx_y,
            'rx_z': rx_z,
            'n_gates': n_gates,
            't_open': t_open,
            't_close': t_close,
            'current': current,
            'waveform_time': waveform_time,
            'waveform_amplitude': waveform_amplitude,
            'n_filters': n_filters,
            'f_cut': f_cut
        })
    
    return pd.DataFrame(table_data)

# Create the measurement table
measurement_df = create_measurement_table(sweeps)
print("Measurement table shape:", measurement_df.shape)
print("\nColumns:", measurement_df.columns.tolist())
print("\nFirst few rows:")
measurement_df.head()

Measurement table shape: (100, 14)

Columns: ['tx_x', 'tx_y', 'tx_z', 'rx_x', 'rx_y', 'rx_z', 'n_gates', 't_open', 't_close', 'current', 'waveform_time', 'waveform_amplitude', 'n_filters', 'f_cut']

First few rows:


,tx_x,tx_y,tx_z,rx_x,rx_y,rx_z,n_gates,t_open,t_close,current,waveform_time,waveform_amplitude,n_filters,f_cut
0,42.060207,4.145819,196.0,42.060207,4.145819,196.0,6,0.000011,0.000030,1.01,"[-0.000674, -0.000660942, -0.000644006, -0.000...","[-0.0, -0.2044, -0.3969, -0.5688, -0.717, -0.8...",4,450000.0
1,42.060207,4.145819,196.0,42.060207,4.145819,196.0,22,0.000022,0.002367,9.99,"[-0.0062, -0.00617761, -0.00615748, -0.0061252...","[-0.0, -0.2595, -0.438, -0.6358, -0.8162, -0.9...",4,450000.0
2,42.060470,4.146311,190.0,42.060470,4.146311,190.0,6,0.000011,0.000030,1.00,"[-0.000674, -0.000660942, -0.000644006, -0.000...","[-0.0, -0.2044, -0.3969, -0.5688, -0.717, -0.8...",4,450000.0
3,42.060470,4.146311,190.0,42.060470,4.146311,190.0,22,0.000022,0.002367,9.99,"[-0.0062, -0.00617761, -0.00615748, -0.0061252...","[-0.0, -0.2595, -0.438, -0.6358, -0.8162, -0.9...",4,450000.0
4,42.060660,4.147010,193.0,42.060660,4.147010,193.0,6,0.000011,0.000030,1.00,"[-0.000674, -0.000660942, -0.000644006, -0.000...","[-0.0, -0.2044, -0.3969, -0.5688, -0.717, -0.8...",4,450000.0
